# CNN implementation from scratch

In [ ]:
import numpy as np

class LinearLayer:
    def __init__(self, D_in, D_out):
        self.D_in = D_in  # input dimension
        self.D_out = D_out  # output dimension
        self.W = np.random.randn(self.D_out, self.D_in)/np.sqrt(D_in)  # W is (D_out, D_in)
        self.b = np.random.randn(self.D_out)/np.sqrt(D_in)  # b is (D_in,)

        # initialize gradients
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def zero_grad(self):
        # reset gradients
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def __call__(self, x):
        """save input for backwards"""
        self.x = x  # (N, D_in) w/ N as the batch size
        return x @ self.W.T + self.b  # output shape (N, D_out)

    def backward(self, d_out):
        """given gradient d_out (N, D_out), return gradient d_in (N, D_in)"""
        self.d_in = d_out @ self.W  # d_in is NOT D_in!
        self.dW += d_out.T @ self.x  # the summation is over samples in a batch
        self.db += d_out.sum(axis=0)
        return self.d_in


In [ ]:
class ReLU:
    def __call__(self, x):
        """save x for backward"""
        self.x = x
        return np.maximum(0, x)
    def backward(self, d_out):
        return d_out * (self.x > 0) # masking

In [ ]:
def im2col(x, kH, kW, stride):
    """
    Each output pixel (i, j) is the inner product of an image patch and the kernel (C_out, C_in*kH*kW);
    flatten input patches to avoid nested for loops
    
    kH: kernel size in height
    kW: kernel size in width
    stride: stride
    input: (N, C_in, H_in, W_in)
    output: (N, H_out*W_out, C_in*kH*kW)
    """
    N, C_in, H_in, W_in = x.shape
    H_out = (H_in - kH)//stride + 1
    W_out = (W_in - kW)//stride + 1

    x_col = []  # x_col to collect all flattened patches
    # iteratre over every pixel
    for i in range(H_out):
        for j in range(W_out):
            x_= x[:, :, i*stride:i*stride+kH, j*stride:j*stride+kW].reshape(N, -1)  # patch (N, C_in*kH*kW)
            x_col.append(x_)

    # trick: np.stack
    return np.stack(x_col, axis=1), H_in, W_in, H_out, W_out  # 1st output: (N, H_out*W_out, C_in*kH*kW)

class Conv2D:
    """
    1. understand the shape of self.W and self.b
    2. we calculate self.dW_col
    """
    def __init__(self, C_in, C_out, kH, kW, stride=1, padding=0):
        self.C_in = C_in
        self.C_out = C_out
        self.stride = stride
        self.padding = padding
        self.W = np.random.randn(C_out, C_in, kH, kW)/np.sqrt(C_in*kH*kW)
        # self.W_col = self.W.reshape(self.C_out, -1)  # (C_out, C_in*kH*kW)
        self.b = np.random.randn(C_out,)
        # self.dW = np.zeros_like(self.W)
        self.dW_col = np.zeros((self.C_out, C_in*kH*kW))  # (C_out, C_in*kH*kW)
        self.db = np.zeros_like(self.b)
        self.kH = kH
        self.kW = kW

    def zero_grad(self):
        self.dW_col = np.zeros((self.C_out, self.C_in*self.kH*self.kW)) 
        self.db = np.zeros_like(self.b)
        if self.dW is not None:
            self.dW = np.zeros_like(self.W)

    def __call__(self, x):
        # x: (N, C_in, H_in, W_in), x_col: (N, H_out*W_out, C_in*kH*kW)
        # save self.x_col for back-prop
        self.x_col, self.H_in, self.W_in, H_out, W_out = im2col(x, self.kH, self.kW, self.stride)

        # calculate output
        self.W_col = self.W.reshape(self.C_out, -1)  # (C_out, C_in*kH*kW)
        output = self.x_col @ self.W_col.T + self.b
        return output.reshape(output.shape[0], H_out, W_out, self.C_out).transpose(0, 3, 1, 2)

    def backward(self, d_out):
        """
        1. d_out is the gradient from (n+1)th layer, which is the input of nth layer in backprop
        2. gradient updates in col is just simple MLP
        3. d_in is the gradient coming out of nth layer; the propagation follows the same formula as forward prop
        4. d_in_col: need to revert im2col to resolve overlapping in col patches
        """
        # 1. d_out: (N, C_out, H_out, W_out)
        N, C_out, H_out, W_out = d_out.shape
        # reshape to fit x_col (N, H_out*W_out, C_in*kH*kW)
        d_out = d_out.reshape(N, C_out, -1)  # (N, C_out, H_out*W_out)

        # 2. gradients of weights
        # the summation is over samples in a batch
        self.dW_col += (d_out @ self.x_col).sum(axis=0)  # (C_out, C_in*kH*kW)
        self.dW = self.dW_col.reshape(self.W.shape)
        self.db += d_out.sum(axis=(0, -1))  # (C_out,)

        # 3. calculate `d_in_col` just like with linear layers
        d_in_col = d_out.transpose(0, 2, 1) @ self.W_col  # (N, H_out*W_out, C_in*kH*kW)

        # 4. d_in_col: need to revert im2col to resolve overlapping in col patches
        d_in_col = d_in_col.reshape(N, H_out*W_out, self.C_in, self.kH, self.kW)
        d_in_col = d_in_col.transpose(0, 2, 1, 3, 4)  # (N, C_in, H_out*W_out, kH, kW)

        # iteratre over every pixel
        d_in = np.zeros((N, self.C_in, self.H_in, self.W_in))
        for i in range(H_out):
            for j in range(W_out):
                d_in[:, :, i*self.stride:i*self.stride+self.kH, j*self.stride:j*self.stride+self.kW] += d_in_col[:, :, i*W_out+j, :, :]
        return d_in


In [ ]:
class MaxPool2D:
    def __init__(self, pool_size):
        # no `self.W` required coz no learning
        self.pool_size = pool_size

    def __call__(self, x):
        N, C, H, W = x.shape
        x = x.reshape(N, C, H//self.pool_size, self.pool_size, W//self.pool_size, self.pool_size)
        self.max_mask = (x == x.max(axis=(3,5), keepdims=True))  # same shape as input x
        x_pooled = x.max(axis=(3, 5))  # (N, C, H_out, W_out)
        return x_pooled

    def backward(self, d_out):
        """
        Each pooled output gradient is broadcast across its whole pooling window.
        Multiplication by max_mask keeps it only at the neuron that produced the maximum; all others receive zero.
        """
        N, C, H_out, W_out = d_out.shape
        d_in = d_out[:, :, :, np.newaxis, :, np.newaxis]  # (N, C, H_out, 1, W_out, 1)
        d_in  = d_in * self.max_mask # broadcast (N, C, H_out, pool_size, W_out, pool_size)
        return d_in.reshape(N, C, H_out*self.pool_size, W_out*self.pool_size)


In [ ]:
class Flatten:
    def __call__(self, x):
        # layer before logits
        self.shape = x.shape
        return x.reshape(x.shape[0], -1)

    def backward(self, d_out):
        return d_out.reshape(self.shape)

In [ ]:
class CrossEntropy:
    def __init__(self, n_class):
        self.n_class = n_class
    def __call__(self, logits, labels):
        """returns the cross entropy loss"""
        # logits (N, 10), label (N, )
        logits = logits - logits.max(axis=-1, keepdims=True)
        z = np.exp(logits).sum(axis=-1, keepdims=True)
        p = (np.exp(logits) / z)
        self.p = p
        return -np.log(p[np.arange(len(labels)), labels]).mean()

    def backward(self, labels):
        one_hot = np.zeros_like(self.p)
        one_hot[np.arange(len(labels)), labels] = 1
        return (self.p - one_hot)/len(labels)

In [ ]:
class Model:
    def __init__(self, n_class, C_in, C_out, kH, kW, pool_size):
        # `ReLU` and `Flatten` saves information of input therefore needs to be in class methods
        self.n_class = n_class
        self.conv = Conv2D(C_in, C_out, kH, kW)
        self.relu = ReLU()
        self.pool = MaxPool2D(pool_size)
        self.flatten = Flatten()
        # run a dummy input to figure out `D_in` for last linear layer
        # here C_in=1, C_out=8, kH=3, kW=3, pool_size=2, stride=1, padding=0
        self.linear = LinearLayer(1352, self.n_class)  

        self.layers = [self.conv, self.relu, self.pool, self.flatten, self.linear]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def backward(self, d_out):
        for layer in self.layers[::-1]:
            d_out = layer.backward(d_out)
        return d_out

    def predict(self, x):
        for layer in self.layers:
            x = layer(x)
        return np.argmax(x, axis=-1)

        


In [ ]:
class SGD:
    def __init__(self, lr):
        self.lr = lr

    def step(self, model):
        for layer in model.layers:
            if hasattr(layer, "W"):
                layer.W -= layer.dW * self.lr
                layer.b -= layer.db * self.lr
                layer.zero_grad()


In [ ]:
def load_mnist():
    import torchvision
    train = torchvision.datasets.MNIST(root='./data', train=True, download=True)
    test = torchvision.datasets.MNIST(root='./data', train=False, download=True)
    X_train = train.data.numpy().reshape(-1, 1, 28, 28).astype(np.float64) / 255.0
    y_train = train.targets.numpy()
    X_test = test.data.numpy().reshape(-1, 1, 28, 28).astype(np.float64) / 255.0
    y_test = test.targets.numpy()
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_mnist()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# training loop

n_class, C_in, C_out, kH, kW, pool_size = 10, 1, 8, 3, 3, 2
lr = 0.01
n_epochs = 1

model = Model(n_class, C_in, C_out, kH, kW, pool_size)
criterion = CrossEntropy(n_class)  # loss function
optimizer = SGD(lr)

batch_size = 64
indices = np.arange(len(X_train))

j = 0
for epoch in range(n_epochs):
    np.random.shuffle(indices)
    for i in range(0, len(X_train), batch_size):
        batch_idx = indices[i:i+batch_size]
        x_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]
        
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        d_out = criterion.backward(y_batch)

        model.backward(d_out)
        optimizer.step(model)

        if (i // batch_size) % 100 == 0:
            lr *= (0.9**(j))
            j += 1
            print(f"step {i}")
            print(f"  train loss, {round(loss, 4)}, learning rate, {round(lr,4)}")

